## 🎯 Learning Objectives
* Design and implement a ReAct agent architecture using LangGraph.
* Integrate external tools into a LangGraph agent workflow.
* Manage conversational memory within a LangGraph agent's state.
* Utilize conditional edges and state updates to control agent execution flow.
* Develop robust agents capable of handling multi-turn interactions and tool usage.


# GS01-L10: Exercise: Build a ReAct Agent in LangGraph with Memory

**Track**: Agentic AI & Automation Tools
**Course**: GS-01 — Building Your First AI Agent with LangGraph
**Section**: Building a Real Agent

## Exercise Task

Your task is to build a ReAct (Reasoning and Acting) agent using LangGraph that can interact with a user, utilize a tool, and maintain conversational memory across multiple turns. The agent should be able to process a user's query, decide whether to use a tool, execute the tool, and then respond to the user, remembering previous interactions.

## Requirements

1.  **LangGraph Integration**: The core agent logic must be implemented using LangGraph's `StateGraph`.
2.  **ReAct Loop**: The agent should exhibit a clear ReAct pattern: `Thought` (LLM decides what to do), `Action` (tool call or final answer), `Observation` (tool output), `Thought` (LLM processes observation), etc.
3.  **Tool Usage**: Integrate at least one external tool. For simplicity, you can use a mock `CalculatorTool` that performs basic arithmetic operations (addition, subtraction, multiplication, division).
4.  **Conversational Memory**: The agent must maintain a history of the conversation (user inputs and agent outputs) and use this memory to inform its responses in subsequent turns.
5.  **Conditional Routing**: Implement conditional edges in LangGraph to decide whether the agent needs to call a tool or provide a final answer.
6.  **Error Handling**: The agent should gracefully handle cases where the tool might fail or return an unexpected output (e.g., by informing the user or attempting a different approach).
7.  **Multi-turn Interaction**: Demonstrate the agent's ability to handle a sequence of related questions where memory is crucial.

## Evaluation Criteria

*   **Correctness**: Does the agent correctly implement the ReAct pattern and use the tool as expected?
*   **Memory Integration**: Does the agent effectively use conversational memory to provide context-aware responses?
*   **LangGraph Structure**: Is the `StateGraph` well-defined with appropriate nodes and edges for the ReAct loop?
*   **Code Clarity**: Is the code well-organized, readable, and commented?
*   **Robustness**: Does the agent handle basic edge cases or tool failures gracefully?
*   **Demonstration**: Provide example interactions that clearly showcase the agent's capabilities, especially its memory and tool usage.


In [ ]:
# Install necessary libraries (if not already installed)
# !pip install -qU langchain langchain_openai langgraph python-dotenv

import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List, Union
import operator

from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, FunctionMessage
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder
from langgraph.graph import StateGraph, END

# Load environment variables from .env file
load_dotenv()

# --- Mock Tool Definition ---
# For this exercise, we'll use a simple mock calculator tool.
# In a real-world scenario, this would be an actual API call or complex function.

@tool
def calculator(operation: str, num1: float, num2: float) -> str:
    """Performs basic arithmetic operations (add, subtract, multiply, divide) on two numbers.
    The 'operation' argument must be one of 'add', 'subtract', 'multiply', or 'divide'.
    Example: calculator(operation='add', num1=5, num2=3)
    """
    try:
        if operation == 'add':
            result = num1 + num2
        elif operation == 'subtract':
            result = num1 - num2
        elif operation == 'multiply':
            result = num1 * num2
        elif operation == 'divide':
            if num2 == 0:
                return "Error: Division by zero is not allowed."
            result = num1 / num2
        else:
            return f"Error: Invalid operation '{operation}'. Choose from 'add', 'subtract', 'multiply', 'divide'."
        return f"The result of {num1} {operation} {num2} is {result}."
    except Exception as e:
        return f"An error occurred during calculation: {e}"

# List of tools available to the agent
# In a real scenario, you might have multiple tools
tools = [calculator]

# --- LLM Initialization ---
# Ensure your OPENAI_API_KEY is set in your .env file or environment variables
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# Bind tools to the LLM for function calling capabilities
llm_with_tools = llm.bind_tools(tools)

# --- Agent State Definition ---
# This defines the state schema for our LangGraph agent.
# It will hold the conversation history.
class AgentState(TypedDict):
    # The 'messages' key is a list of BaseMessage objects, representing the conversation history.
    # The 'operator.add' function is used to append new messages to the list.
    messages: Annotated[List[BaseMessage], operator.add]


## Your Implementation

Now it's your turn! Using the provided setup code (tools, LLM, and `AgentState` definition), implement the ReAct agent with memory using LangGraph. 

Follow the requirements outlined above. Your solution should include:

1.  **Agent Nodes**: Functions that represent the agent's steps (e.g., calling the LLM, executing a tool).
2.  **Conditional Logic**: A function to determine the next step based on the LLM's output (e.g., call tool or finish).
3.  **Graph Construction**: Use `StateGraph` to define the workflow, including entry point, nodes, and edges.
4.  **Execution**: Demonstrate the agent's functionality with a series of multi-turn interactions that showcase its memory and tool-use capabilities.


In [ ]:
# --- Reference Solution ---

# 1. Define Agent Nodes

# Node to call the LLM and get its response
def call_llm(state: AgentState) -> dict:
    """Invokes the LLM with the current conversation history and returns the LLM's response."""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Node to execute a tool based on the LLM's decision
def call_tool(state: AgentState) -> dict:
    """Executes the tool suggested by the LLM and returns the tool's output as a FunctionMessage."""
    messages = state['messages']
    last_message = messages[-1]

    # Check if the last message contains tool_calls
    if not last_message.tool_calls:
        # This should ideally not happen if routing is correct, but good for robustness
        return {"messages": [AIMessage(content="Error: LLM did not suggest a tool call.")]}

    # Iterate through all tool calls suggested by the LLM
    tool_outputs = []
    for tool_call in last_message.tool_calls:
        tool_name = tool_call['name']
        tool_args = tool_call['args']

        # Find the corresponding tool function
        selected_tool = next((t for t in tools if t.name == tool_name), None)

        if selected_tool:
            try:
                # Execute the tool with the provided arguments
                output = selected_tool.invoke(tool_args)
                tool_outputs.append(FunctionMessage(content=str(output), name=tool_name))
            except Exception as e:
                tool_outputs.append(FunctionMessage(content=f"Tool '{tool_name}' failed: {e}", name=tool_name))
        else:
            tool_outputs.append(FunctionMessage(content=f"Tool '{tool_name}' not found.", name=tool_name))

    return {"messages": tool_outputs}

# 2. Define Conditional Logic

# This function determines the next step in the graph based on the LLM's output.
def should_continue(state: AgentState) -> str:
    """Determines whether the agent should continue by calling a tool or end the conversation."""
    messages = state['messages']
    last_message = messages[-1]
    # If the LLM's last message has tool calls, it means it wants to use a tool.
    if last_message.tool_calls:
        return "continue"
    # Otherwise, the LLM has provided a final answer.
    else:
        return "end"

# 3. Graph Construction

# Create a new StateGraph instance with our defined AgentState.
workflow = StateGraph(AgentState)

# Add nodes to the graph
workflow.add_node("llm", call_llm)       # Node for calling the LLM
workflow.add_node("tool", call_tool)     # Node for calling a tool

# Set the entry point of the graph
workflow.set_entry_point("llm")

# Add conditional edges
# From the 'llm' node, decide whether to go to 'tool' or 'END'
workflow.add_conditional_edges(
    "llm",           # Source node
    should_continue, # Function to determine the next node
    {
        "continue": "tool", # If 'should_continue' returns "continue", go to 'tool' node
        "end": END          # If 'should_continue' returns "end", terminate the graph
    }
)

# Add a normal edge from the 'tool' node back to the 'llm' node.
# After a tool is executed, the agent should always go back to the LLM
# to process the tool's output and decide the next step (ReAct loop).
workflow.add_edge('tool', 'llm')

# Compile the graph into a runnable agent
app = workflow.compile()

# 4. Demonstration

print("\n--- Agent Demonstration ---")

# Example 1: Simple question without tool usage
print("\nUser: Hello, how are you today?")
inputs = {"messages": [HumanMessage(content="Hello, how are you today?")]}
for s in app.stream(inputs):
    if "llm" in s:
        print(f"LLM Thought: {s['llm']['messages'][-1].content}")
    elif "__end__" in s:
        print(f"Agent Response: {s['__end__']['messages'][-1].content}")

# Example 2: Question requiring tool usage
print("\nUser: What is 15 multiplied by 7?")
inputs = {"messages": [HumanMessage(content="What is 15 multiplied by 7?")]}
for s in app.stream(inputs):
    if "llm" in s:
        print(f"LLM Thought/Action: {s['llm']['messages'][-1].content}")
    elif "tool" in s:
        print(f"Tool Observation: {s['tool']['messages'][-1].content}")
    elif "__end__" in s:
        print(f"Agent Response: {s['__end__']['messages'][-1].content}")

# Example 3: Multi-turn interaction with memory and tool usage
print("\nUser: What is the sum of 123 and 456?")
# We need to maintain the state across turns for memory
current_state = {"messages": [HumanMessage(content="What is the sum of 123 and 456?")]}
for s in app.stream(current_state):
    if "llm" in s:
        print(f"LLM Thought/Action: {s['llm']['messages'][-1].content}")
    elif "tool" in s:
        print(f"Tool Observation: {s['tool']['messages'][-1].content}")
    elif "__end__" in s:
        print(f"Agent Response: {s['__end__']['messages'][-1].content}")
        current_state = s['__end__'] # Update state for the next turn

print("\nUser: And what is that result divided by 3?")
# The agent should remember the previous result from the 'current_state'
# LangGraph automatically passes the full state, including previous messages, to the next invocation.
# We just need to append the new HumanMessage to the existing messages.
current_state['messages'].append(HumanMessage(content="And what is that result divided by 3?"))
for s in app.stream(current_state):
    if "llm" in s:
        print(f"LLM Thought/Action: {s['llm']['messages'][-1].content}")
    elif "tool" in s:
        print(f"Tool Observation: {s['tool']['messages'][-1].content}")
    elif "__end__" in s:
        print(f"Agent Response: {s['__end__']['messages'][-1].content}")
        current_state = s['__end__'] # Update state for the next turn

# Example 4: Tool error handling
print("\nUser: Divide 10 by 0.")
inputs = {"messages": [HumanMessage(content="Divide 10 by 0.")]}
for s in app.stream(inputs):
    if "llm" in s:
        print(f"LLM Thought/Action: {s['llm']['messages'][-1].content}")
    elif "tool" in s:
        print(f"Tool Observation: {s['tool']['messages'][-1].content}")
    elif "__end__" in s:
        print(f"Agent Response: {s['__end__']['messages'][-1].content}")
